Email in AJO: [Week27] Email - LOY-00304 - CineClub 5+ Unused Tickets_May13

Context:
•	On May 13, sent an email to guests with 5+ saved CineClub tickets.
•	While there are 50K+ guests in this audience, we sent as a test to 10% of the audience to see how it impacts ticket use, signups and churn

There were 4 versions of the email:
1.	Active member + has visited in P12M
2.	Active member + has NOT visited in P12M
3.	Cancelled member + has visited in P12M
4.	Cancelled member + has NOT visited in P12M

Post reporting requirements:
•	Key KPIs:
o	All - Drive use of saved tickets
•	Compare to guests who didn’t get the email
o	For Active - understand the impact on churn/exits (i.e. if they realize they have tons of saved tickets, they might cancel) – how many cancelled
•	Compare to guests who didn’t get the email
•	Secondary KPIs
o	For NOT visited in P12M – did it drive return visits to Cineplex
•	Compare to guests who didn’t get the email
o	For Cancelled members - see if this leads to returning signups to the program
•	Compare to guests who didn’t get the email
•	Timeframe - see how these KPIs were impacted within 30 days after sending the email (May 13 – June 13)

In [0]:
%sql
WITH sent_emails AS (
  SELECT
    CASE journey_action_id
      WHEN 'bd448f28-0211-4e36-95f8-2e946c97fc26' THEN 'English - Active - Visit P1Y'
      WHEN '3cbba23f-a437-48f7-99dc-dee74a7c4fd6' THEN 'English - Active - No Visit P1Y'
      WHEN 'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94' THEN 'English - Cancelled - Visit P1Y'
      WHEN '991d60ac-7e3c-44a0-b7b1-b16afae1281d' THEN 'English - Cancelled - No Visit P1Y'
    END AS journey_label,
    Email_Address,
    message_profile_id
  FROM cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback
  WHERE journey_action_id IN (
      'bd448f28-0211-4e36-95f8-2e946c97fc26',
      '3cbba23f-a437-48f7-99dc-dee74a7c4fd6',
      'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94',
      '991d60ac-7e3c-44a0-b7b1-b16afae1281d'
    )
    AND feedback_status = 'sent'
)
Select journey_label,Count(*) from sent_emails group by journey_label

In [0]:
from pyspark.sql import functions as F

journey_map = {
    'bd448f28-0211-4e36-95f8-2e946c97fc26': 'English - Active - Visit P1Y',
    '3cbba23f-a437-48f7-99dc-dee74a7c4fd6': 'English - Active - No Visit P1Y',
    'e24b9c3d-cdc7-4cc5-9676-ea773afe0b94': 'English - Cancelled - Visit P1Y',
    '991d60ac-7e3c-44a0-b7b1-b16afae1281d': 'English - Cancelled - No Visit P1Y',
}

journey_label = F.create_map([F.lit(x) for kv in journey_map.items() for x in kv])

sent_emails = (
    spark.table("cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback")
    .filter(F.col("journey_action_id").isin(list(journey_map.keys())))
    .filter(F.col("feedback_status") == "sent")
    .select(
        journey_label[F.col("journey_action_id")].alias("journey_label"),
        "Email_Address",
        "message_profile_id",
    )
)

In [0]:
sent_emails.groupBy("journey_label").count().display()

In [0]:
T_emails = spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address")

T_cde_ids = (
    sent_emails
    .join(T_emails, "Email_Address", how ="inner" )
    .select("journey_label", "CDE_ID")
    .distinct()
)


In [0]:
from pyspark.sql import functions as F

# Check for fan-out risk in the email -> CDE_ID join
join_check = (
    sent_emails
    .groupBy("journey_label")
    .agg(F.count("*").alias("sent_rows"))
)

cde_check = (
    T_cde_ids
    .groupBy("journey_label")
    .agg(
        F.count("*").alias("joined_rows"),
        F.countDistinct("CDE_ID").alias("distinct_cde_ids")
    )
)

join_check.join(cde_check, "journey_label").show(truncate=False)

In [0]:
df_snapshot= spark.table('cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot')

df_holdout = (
    df_snapshot
    .filter(F.col("Snapshot_Date") == F.lit("20260513"))
    .filter(F.col("recognitions_left") >= 5)
)